In [1]:
import utils
import os

np = utils.importer('numpy')
plt = utils.importer('matplotlib.pyplot')
pickle = utils.importer('pickle')
meshio = utils.importer('meshio')
Silhouette, Landscape = utils.importer('gudhi.representations', 'Silhouette', 'Landscape')
LogisticRegression = utils.importer('sklearn.linear_model', 'LogisticRegression')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.inspection import DecisionBoundaryDisplay

In [2]:
malepath = 'E:/TDA/Male/'
male_scans = os.listdir(malepath)
male_nb_scans = len(male_scans)

In [3]:
girlpath = 'E:/TDA/Female/'
girl_scans = os.listdir(girlpath)
girl_nb_scans = len(girl_scans)

all_scans = male_scans + girl_scans
nb_scans = male_nb_scans + girl_nb_scans

In [4]:
def scan_from_index(i):
    if i < male_nb_scans:
        path = malepath + str(all_scans[i])
    else:
        path = girlpath + str(all_scans[i])
    scan = meshio.read(path)
    scan = scan.points
    return scan

In [5]:
def plot_scan_3D_interactive(i):
    scan = scan_from_index(i)
    points_set = utils.point_set(scan)
    points_set.plot(color='green', opacity=0.5, size=2)

In [6]:
with open ('./Data/XY_decolor.bin', 'rb') as fp:
    XY_decolor = pickle.load(fp)

with open ('./Data/XX_decolor.bin', 'rb') as fp:
    XX_decolor = pickle.load(fp)

X_decolor = XY_decolor + XX_decolor
del XX_decolor, XY_decolor

In [7]:
test0 = [xd[0][:-1] for xd in X_decolor]
test1 = [xd[1] for xd in X_decolor]
test2 = [xd[2] for xd in X_decolor]

In [8]:
n0, n1, n2 = 1000, 10000, 10000
weight = lambda x: np.sqrt(x[0])
SH0 = Silhouette(resolution=n0)
sh0 = SH0.fit_transform(test0)

SH1 = Silhouette(resolution=n1, weight=weight)
sh1 = SH1.fit_transform(test1)

SH2 = Silhouette(resolution=n2, weight=weight)
sh2 = SH2.fit_transform(test2)

In [9]:
n = 210
m = (n0+n1+n2)//n
# m = n2//n

temp = np.array([list(x0) + list(x1) + list(x2)
                for x0, x1, x2 in zip(sh0, sh1, sh2)])
# temp = sh2
X = np.array([[np.mean(x[n*i:n*i+n]) for i in range(m)] for x in temp])
X = StandardScaler().fit_transform(X)

y = [i < male_nb_scans for i in range(nb_scans)]


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/3, random_state=42)

In [11]:
with open ('./Data/MPI/X_decolor.bin', 'rb') as fp:
    X_decolor_ = pickle.load(fp)

In [12]:
test0_ = [xd[0][:-1] for xd in X_decolor_]
test1_ = [xd[1] for xd in X_decolor_]
test2_ = [xd[2] for xd in X_decolor_]

In [13]:
SH0 = Silhouette(resolution=n0)
sh0_ = SH0.fit_transform(test0_)

SH1 = Silhouette(resolution=n1, weight=weight)
sh1_ = SH1.fit_transform(test1_)

SH2 = Silhouette(resolution=n2, weight=weight)
sh2_ = SH2.fit_transform(test2_)

In [14]:
temp_ = np.array([list(x0) + list(x1) + list(x2)
                 for x0, x1, x2 in zip(sh0_, sh1_, sh2_)])
# temp_ = sh2_
X_ = np.array([[np.mean(x_[n*i:n*i+n]) for i in range(m)] for x_ in temp_])
X_ = StandardScaler().fit_transform(X_)

In [15]:
names = [
    "Nearest Neighbors",
    "Linear SVM",
    "RBF SVM",
    "Gaussian Process",
    "Decision Tree",
    "Random Forest",
    "Neural Net",
    "AdaBoost",
    "Naive Bayes",
    "Logistic Regression"
]

classifiers = [
    KNeighborsClassifier(),
    SVC(kernel="linear"),
    SVC(gamma=2),
    GaussianProcessClassifier(),
    DecisionTreeClassifier(),
    RandomForestClassifier(n_estimators=50, random_state=42),
    MLPClassifier(max_iter=10000),
    AdaBoostClassifier(),
    GaussianNB(),
    LogisticRegression(max_iter=10000)
]

In [16]:
scores = []

for name, clf in zip(names, classifiers):
    clf.fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    score_train = clf.score(X_train, y_train)
    scores.append(score)
    print(name, ' -  Score:', score, ' -  STRAIN:', score_train)

Nearest Neighbors  -  Score: 0.8868110236220472  -  STRAIN: 0.922244094488189
Linear SVM  -  Score: 0.9212598425196851  -  STRAIN: 0.9104330708661418
RBF SVM  -  Score: 0.6230314960629921  -  STRAIN: 1.0
Gaussian Process  -  Score: 0.8956692913385826  -  STRAIN: 0.984744094488189
Decision Tree  -  Score: 0.8533464566929134  -  STRAIN: 1.0
Random Forest  -  Score: 0.9212598425196851  -  STRAIN: 1.0
Neural Net  -  Score: 0.937007874015748  -  STRAIN: 1.0
AdaBoost  -  Score: 0.922244094488189  -  STRAIN: 0.9360236220472441
Naive Bayes  -  Score: 0.6368110236220472  -  STRAIN: 0.6387795275590551
Logistic Regression  -  Score: 0.9104330708661418  -  STRAIN: 0.9015748031496063


In [17]:
Y_ = classifiers[np.argmax(scores)].predict(X_)

for i, y in enumerate(Y_):
    if not y:
        print('%4d' % i, end=' ')

   8   24   25   26   27   28   36   42   45   46   53   56   58   59   61   63   64   65   68   70   76   77   78   79   81   83   86   88  103  116  118  120  123  131  132  138  139  141  147  155  160  162  165  171  175  181  183  185  189  193  201  203  205  207  208  213  225  231  233  241  246  249  250  251  257  261  267  269  270  271  272  273  275  276  282  283  289  290  291  292  294  296  297  302  304  305  306  339  346  348  350  353  354  357  360  361  364  366  367  368  373  374  397  402  409  429  432  447  454  456  457  460  461  463  465  468  470  476  478  483  486  489  490  492  494  495  496  507  509  512  515  517  518  521  527  528  532  537  538  540  541  556  557  560  567  569  572  575  576  577  579  582  586  592  595  597  598  602  604  612  613  616  619  620  622  627  629  630  631  632  633  634  635  637  640  644  649  650  654  655  656  658  659  662  665  668  672  673  684  685  688  690  691  697  702  703  705  710  712  716 